In [0]:
# Import Libraries 
from pyspark.sql.functions import *

In [0]:
# Load Bronze Billing Table 
billing = spark.table("billing_transactions_300_k")

In [0]:
# View Data 
display(billing)

TransactionID,CustomerID,TransactionDate,Revenue,Product
T0,139260,2024-02-06,2291.66,Plan B
T1,193679,2025-04-12,4997.86,Plan B
T2,131483,2025-12-12,2892.57,Plan A
T3,118325,2024-10-01,2082.69,Plan A
T4,180032,2025-02-21,2756.25,Plan C
T5,127085,2025-06-22,2263.74,Plan B
T6,196622,2025-07-21,7108.16,Plan B
T7,149639,2025-06-24,3358.92,Plan A
T8,173360,2024-09-18,1776.46,Plan C
T9,196749,2024-12-11,3802.08,Plan B


In [0]:
# Print schema 
billing.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Product: string (nullable = true)



In [0]:
# Check Row Count 
print("Total Billing Records:", billing.count())

Total Billing Records: 300000


In [0]:
# Check Missing Values
billing.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in billing.columns
]).show()

+-------------+----------+---------------+-------+-------+
|TransactionID|CustomerID|TransactionDate|Revenue|Product|
+-------------+----------+---------------+-------+-------+
|            0|         0|              0|      0|      0|
+-------------+----------+---------------+-------+-------+



In [0]:
# Remove Duplicate Transactions (Each transaction should appear only once) 
silver_billing = billing.dropDuplicates(["TransactionID"])

In [0]:
# print("Original Records:", billing.count())

print("After Removing Duplicates:", silver_billing.count())

After Removing Duplicates: 300000


In [0]:
# Handle Missing Values (Replace missing revenue with 0) 
silver_billing = silver_billing.fillna({
    "Revenue":0
})

In [0]:
# Replace missing Product 
silver_billing = silver_billing.fillna({
    "Product":"UNKNOWN"
})

In [0]:
# Standardize Product Names (Remove extra spaces) 
silver_billing = silver_billing.withColumn(
    "Product",
    trim(col("Product"))
)

In [0]:
# Convert to uppercase 
silver_billing = silver_billing.withColumn(
    "Product",
    upper(col("Product"))
)

In [0]:
# Convert Transaction Date (Current datatype) 
silver_billing = silver_billing.withColumn(
    "TransactionDate",
    to_date(col("TransactionDate"))
)

In [0]:
silver_billing.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Revenue: double (nullable = false)
 |-- Product: string (nullable = false)



In [0]:
# Validate Revenue

# Revenue cannot be negative.

# Remove invalid transactions. 
silver_billing = silver_billing.filter(
    col("Revenue") >= 0
)

In [0]:
# Create Billing Month

# Power BI usually analyzes revenue monthly.

# Create Month column. 

silver_billing = silver_billing.withColumn(
    "BillingMonth",
    date_format(col("TransactionDate"),"yyyy-MM")
)

In [0]:
# Create Monthly Billing Summary

# Now aggregate the transaction-level data to the customer-month level. 
silver_billing = silver_billing.groupBy(
    "CustomerID",
    "BillingMonth"
).agg(
    sum("Revenue").alias("MonthlyRevenue"),
    count("TransactionID").alias("TransactionCount")
)

In [0]:
# Verify Output 
display(silver_billing)

CustomerID,BillingMonth,MonthlyRevenue,TransactionCount
139260,2024-02,9302.369999999999,3
193679,2025-04,4997.86,1
131483,2025-12,2892.57,1
118325,2024-10,2082.69,1
180032,2025-02,2756.25,1
127085,2025-06,2263.74,1
196622,2025-07,7108.16,1
149639,2025-06,3358.92,1
173360,2024-09,1776.46,1
196749,2024-12,3802.08,1


In [0]:
# Validate Schema  
# There is no Revenue column anymore because it has been aggregated into MonthlyRevenue.
silver_billing.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- BillingMonth: string (nullable = true)
 |-- MonthlyRevenue: double (nullable = true)
 |-- TransactionCount: long (nullable = false)



In [0]:
# Save Silver Billing 
silver_billing.write.mode("overwrite").saveAsTable("silver_billing")

In [0]:
# Validate Saved Table 
display(spark.table("silver_billing"))

CustomerID,BillingMonth,MonthlyRevenue,TransactionCount
145718,2025-02,656.74,1
175833,2024-08,4821.14,1
139492,2025-03,3843.37,1
107192,2025-02,2388.86,1
181030,2025-05,3233.65,1
178137,2025-05,4029.18,1
160059,2025-12,3717.62,1
123619,2024-06,2515.23,1
149727,2024-08,4616.64,1
180974,2025-11,2235.6,1
